# Lab 03 - Notebooks Avanzados

**Objetivos:**
- Dominar magic commands
- Usar widgets para parametrización
- Comunicación entre notebooks
- Debugging y logging

## Ejercicio 1: Magic Commands

### Markdown

In [ ]:
%md
# Título usando Magic Command
Este es un **markdown** usando `%md`

- Item 1
- Item 2
- Item 3

### Crear datos de prueba

In [ ]:
# Crear DataFrame de ventas
from pyspark.sql.functions import *
from datetime import datetime, timedelta
import random

# Datos de ejemplo
products = ["Laptop", "Mouse", "Keyboard", "Monitor", "Webcam"]
regions = ["North", "South", "East", "West"]

data = []
for i in range(100):
    data.append({
        "sale_id": f"S{i:04d}",
        "date": (datetime.now() - timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d"),
        "product": random.choice(products),
        "region": random.choice(regions),
        "amount": round(random.uniform(100, 2000), 2),
        "quantity": random.randint(1, 5)
    })

df_sales = spark.createDataFrame(data)
df_sales.createOrReplaceTempView("sales")

print(f"✅ Datos creados: {df_sales.count()} ventas")
display(df_sales.limit(5))

### SQL Query

In [ ]:
%sql
SELECT 
    region,
    product,
    COUNT(*) as num_sales,
    ROUND(SUM(amount), 2) as total_revenue,
    ROUND(AVG(amount), 2) as avg_amount
FROM sales
GROUP BY region, product
ORDER BY total_revenue DESC
LIMIT 10

### Shell Commands

In [ ]:
%sh
echo "📁 Listar directorio actual:"
pwd
ls -lh

### File System Commands

In [ ]:
%fs ls /tmp/

## Ejercicio 2: Widgets (Parametrización)

In [ ]:
# Crear widgets
dbutils.widgets.text("fecha_inicio", "2026-05-01", "📅 Fecha Inicio")
dbutils.widgets.dropdown("region", "All", ["All", "North", "South", "East", "West"], "🌍 Región")
dbutils.widgets.combobox("producto", "All", ["All"] + products, "📦 Producto")
dbutils.widgets.multiselect("metricas", "amount,quantity", ["amount", "quantity", "sale_id"], "📊 Métricas")

print("✅ Widgets creados")

In [ ]:
# Leer valores de widgets
fecha_inicio = dbutils.widgets.get("fecha_inicio")
region_filtro = dbutils.widgets.get("region")
producto_filtro = dbutils.widgets.get("producto")
metricas = dbutils.widgets.get("metricas").split(",")

print("📋 Parámetros seleccionados:")
print(f"  Fecha inicio: {fecha_inicio}")
print(f"  Región: {region_filtro}")
print(f"  Producto: {producto_filtro}")
print(f"  Métricas: {metricas}")

In [ ]:
# Aplicar filtros dinámicamente
df_filtered = df_sales.filter(col("date") >= fecha_inicio)

if region_filtro != "All":
    df_filtered = df_filtered.filter(col("region") == region_filtro)
    
if producto_filtro != "All":
    df_filtered = df_filtered.filter(col("product") == producto_filtro)

print(f"📊 Registros después de filtrar: {df_filtered.count()}")
display(df_filtered)

## Ejercicio 3: Debugging y Logging

In [ ]:
import time
from functools import wraps

def time_it(func):
    """Decorator para medir tiempo de ejecución"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"⏱️  {func.__name__} ejecutado en {end - start:.2f} segundos")
        return result
    return wrapper

@time_it
def procesar_datos(df):
    """Simula procesamiento de datos"""
    result = df.groupBy("region", "product").agg(
        sum("amount").alias("total"),
        avg("amount").alias("promedio")
    ).collect()
    return result

# Ejecutar
resultado = procesar_datos(df_sales)
print(f"✅ Procesado {len(resultado)} grupos")

In [ ]:
# Función para detectar columnas con nulos
def find_null_columns(df):
    """Encuentra columnas con valores nulos"""
    null_counts = []
    
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            null_counts.append((col_name, null_count))
    
    return null_counts

# Probar
nulls = find_null_columns(df_sales)
if nulls:
    print("⚠️  Columnas con nulos:")
    for col_name, count in nulls:
        print(f"  - {col_name}: {count} nulos")
else:
    print("✅ No hay valores nulos")

In [ ]:
# Analizar plan de ejecución
df_complex = df_sales \
    .filter(col("amount") > 500) \
    .groupBy("region") \
    .agg(sum("amount").alias("total"))

print("📊 Plan de ejecución físico:")
df_complex.explain()

## Ejercicio 4: Guardar Resultados

In [ ]:
# Guardar resultados del análisis
output_path = "/tmp/lab03/sales_analysis"

df_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .save(output_path)

print(f"✅ Resultados guardados en: {output_path}")

In [ ]:
# Verificar particiones creadas
files = dbutils.fs.ls(output_path)
partitions = [f.name for f in files if f.name.startswith("region=")]

print(f"📁 Particiones creadas: {len(partitions)}")
for p in partitions:
    print(f"  - {p}")

## ✅ Lab Completado

Has aprendido:
- ✅ Magic commands (%md, %sql, %sh, %fs)
- ✅ Widgets para parametrización
- ✅ Debugging con decorators y timing
- ✅ Análisis de planes de ejecución
- ✅ Particionado de datos